2033185


In [2]:
### Identify all input genomes
combined_cluster_data = pl.read_csv('combined_cluster_data_w_urls.tsv', separator='\t')
print(combined_cluster_data.height)
print(combined_cluster_data.unique('genome').height)

2054956
2054956


In [3]:
### Identify genomes that have been downloaded
downloaded_genomes_join = downloaded_genomes.join(combined_cluster_data, on='genome', how='inner')
downloaded_genomes_join.height
downloaded_genomes_join.rename({'column_1': 'url'}).write_csv('downloaded_genomes_w_urls.tsv', separator='\t')

In [4]:
### Identify genomes that have yet to be downloaded
not_downloaded_genomes = combined_cluster_data.filter(~pl.col('genome').is_in(set(downloaded_genomes['genome'])))

In [ ]:
### Write out url file for cleanup downloads
not_downloaded_genomes[['column_1']].write_csv('download_cleanup_1_urls.txt', include_header=False)

In [ ]:
%%bash
### Cleanup download
aria2c \
    --input-file=download_cleanup_1_urls.txt \
    --dir=cleanup_1_genomes \
    --max-concurrent-downloads=16 \
    --continue \
    --max-tries=5 \
    --retry-wait=5
    

In [5]:
### Identify largest genera to start compressing
downloaded_genomes_join.group_by('genus').len().sort('len', descending=True).head(10)

genus,len
str,u32
"""Salmonella""",331067
"""Escherichia""",175013
"""Bacteroides""",62098
"""Campylobacter_D""",59220
"""Alistipes""",46845
"""Streptococcus""",44612
"""Phocaeicola""",42225
"""Agathobacter""",41773
"""Bifidobacterium""",39986


In [57]:
(
    downloaded_genomes_join
        .filter(pl.col('genus') == 'Staphylococcus')
        .sort('adjusted_quality_score', descending=True)
        .tail(-1)
        [['path']]
        .write_csv('agc_inputs/Staphyloccocus.txt', include_header=False, separator='\t')
)

In [ ]:
%%bash
### Run AGC on Staphyloccocus
# -a -b and -s taken from publication
# -f 0.01 value taken from GitHub README
time agc create \
    /gscratch/scrubbed/carsonjm/2026.03.02/99/a980661c0d8ab0d913a4f2f2273e2c/host_fastas/RSGB23-1_GCA-016886045-V1_GENO_10000001.fa.gz \
    -i agc_inputs/Staphyloccocus.txt \
    -a true \
    -b 500 \
    -s 1500 \
    -d false \
    -o agc_archives/Staphylococcus.agc \
    -t 4 \
    -f 0.01 \
    -v 1
### GZ Fasta stats
# ~750 Kb per compressed Fasta
# ~28.5 GB for 38,000 compressed Fastas

### AGC stats
# __ GB
# __ time to compress (4 CPUs)

In [ ]:
(
    downloaded_genomes_join
        .filter(pl.col('genus') == 'Staphylococcus')
        .sort('adjusted_quality_score', descending=True)
        [['genome', 'path']]
        .write_csv('sketchlib_inputs/Staphyloccocus.tsv', include_header=False, separator='\t')
)

In [64]:
(
    downloaded_genomes_join
        .sort('adjusted_quality_score', descending=True)
        .filter(pl.col('genus') == 'Staphylococcus')
        [['genome']]
        .write_csv('sketchlib_inputs/Staphylococcus_objects.tsv', include_header=False, separator='\t')
)

In [ ]:
%%bash
### Test sketchlib on staph
sketchlib sketch \
    -f sketchlib_inputs/Staphyloccocus.tsv \
    -o sketchlib_inputs/Staphyloccocus_s1000_k21 \
    -k 21 \
    -s 1000 \
    -v \
    --threads 32

In [ ]:
%%bash
### Run all-v-all sketchlib on Staph
sketchlib dist \
    sketchlib_inputs/Staphyloccocus_s1000_k21.skm \
    -o sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100.tsv \
    -k 21 \
    --ani \
    --threads 32 \
    --verbose \
    --knn 100

# Add header line to file
sed -i '1i genome1\tgenome2\tani' sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100.tsv

In [ ]:
%%bash
### Extract derep reps using clusty (99.9% ANI)
clusty \
    sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100.tsv \
    sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100_clusty0_999_cdhit.tsv \
    --objects-file sketchlib_inputs/Staphylococcus_objects.tsv \
    --similarity \
    --min ani 0.999 \
    --out-representatives
# This yields 1,117 clusters

In [ ]:
%%bash
### Extract derep reps using clusty (99.99% ANI)
clusty \
    sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100.tsv \
    sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100_clusty0_9999_cdhit.tsv \
    --objects-file sketchlib_inputs/Staphylococcus_objects.tsv \
    --similarity \
    --min ani 0.9999 \
    --out-representatives
# This yields 24,284 clusters

In [ ]:
%%bash
### Extract derep reps using clusty (99.999% ANI)
clusty \
    sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100.tsv \
    sketchlib_inputs/Staphyloccocus_s1000_k21_dist_knn100_clusty0_9999_cdhit.tsv \
    --objects-file sketchlib_inputs/Staphylococcus_objects.tsv \
    --similarity \
    --min ani 0.99999 \
    --out-representatives
# This yields 32,987 clusters
# Any higher minimum also yields 32,987

In [ ]:
### Pipeline:
# ALL
# Split genomes by genus

# WITHIN GENERA
# Download URLS (paths also included)
# Create sketchlib sketch of genomes (within genera)
# Compare newly sketched genomes vs current genus sketch (if available, otherwise vs self)
# Run clusty to identify unique (99.999%), strain (99.99%) and genomovar (99.9%) reps
# Compress unique reps of --min ani 0.99999 using AGC (append if exists, create if not)
# Add new unique reps to genus sketch
# Cleanup downloaded genomes (if not local path)